## Use of the Claude API

Reference: https://platform.claude.com/docs/en/cli-sdks-libraries/sdks/python

### API key check

Loads `ANTHROPIC_API_KEY` from `.env` and confirms it's set before making any calls.

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise ValueError("ANTHROPIC_API_KEY environment variable not set")

### Basic message creation, with error handling

Example usage from the Claude platform docs. Catches connection errors, rate limiting (429), and other non-200 API status errors separately so each can be handled appropriately.

In [ ]:
import os
from anthropic import Anthropic
import anthropic

client = Anthropic(
    # This is the default and can be omitted
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
)

try:
    message = client.messages.create(
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": "Hello, Claude",
            }
        ],
        model="claude-opus-4-8",
    )

    for block in message.content:
        if block.type == "text":
            print(block.text)

except anthropic.APIConnectionError as e:
    print("The server could not be reached")
    print(e.__cause__)  # an underlying Exception, likely raised within httpx
except anthropic.RateLimitError as e:
    print("A 429 status code was received; we should back off a bit.")
    print(e.status_code)
except anthropic.APIStatusError as e:
    print("Another non-200-range status code was received")
    print(e.status_code)
    print(e.response)

### Token counting and usage

Counts input tokens before sending a request (`messages.count_tokens`), then prints the actual `usage` returned with the response for comparison.

In [ ]:
import os
from anthropic import Anthropic

client = Anthropic(
    # This is the default and can be omitted
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
)

message = client.messages.create(
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Hello, Claude",
        }
    ],
    model="claude-opus-4-8",
)

count = client.messages.count_tokens(
    model="claude-opus-4-8", messages=[{"role": "user", "content": "Hello, world"}]
)
print(count.input_tokens)  # 10

for block in message.content:
    if block.type == "text":
        print(block.text)

print(message.usage)
# Usage(input_tokens=25, output_tokens=13)

### Tool use via `@beta_tool`

The `@beta_tool` decorator generates a tool's JSON schema from the function's signature and docstring. `client.beta.messages.tool_runner` then drives the tool-call loop automatically — no manual dispatch of tool calls needed.

In [ ]:
import json
from anthropic import Anthropic, beta_tool

client = Anthropic()


@beta_tool
def get_weather(location: str) -> str:
    """Get the weather for a given location.

    Args:
        location: The city and state, for example, San Francisco, CA
    Returns:
        A JSON-encoded string with the location, temperature, and weather condition.
    """
    return json.dumps(
        {
            "location": location,
            "temperature": "68°F",
            "condition": "Sunny",
        }
    )


runner = client.beta.messages.tool_runner(
    max_tokens=1024,
    model="claude-opus-4-8",
    tools=[get_weather],
    messages=[
        {"role": "user", "content": "What is the weather in SF?"},
    ],
)
for message in runner:
    print(message)